# Step 4A.1 — аудит аномалії negative lag + Step 4A.2 — temporal CVE forecast dataset builder
Частина 1: розбирає 279 CVE, де `kev_date_added < published` (не видаляє, лише категоризує й
шукає патерн — можливий зв'язок із запуском каталогу KEV 3 листопада 2021 р., коли одразу
додано 306 історичних вразливостей).

Частина 2: будує датасет для незалежного (не циркулярного) forecasting CVE-рівня:
**одна CVE — один рядок** (не панель по 65 знімках), з точкою відліку `t` = перший знімок
EPSS на момент або одразу після `published`. Це структурно унеможливлює появу тієї самої
CVE одночасно в train/val/test (на відміну від панельного дизайну по всіх знімках).

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CTI_DIR = Path("/srv/ids_research/data/cti")
master = pd.read_parquet(CTI_DIR / "cve_master.parquet")
master["published"] = pd.to_datetime(master["published"], errors="coerce")
master["kev_date_added"] = pd.to_datetime(master.get("kev_date_added"), errors="coerce")
TODAY = pd.Timestamp("2026-09-22")

snaps = sorted((CTI_DIR / "epss" / "history").glob("epss_scores-*.csv.gz"))
snap_dates = sorted(pd.Timestamp(p.name[12:22]) for p in snaps)
KEV_LAUNCH = pd.Timestamp("2021-11-03")   # https://www.cisa.gov ... запуск каталогу, 306 записів одразу
print(f"cve_master: {len(master):,} CVE | знімків EPSS: {len(snap_dates)} ({snap_dates[0].date()}—{snap_dates[-1].date()})")

cve_master: 395,739 CVE | знімків EPSS: 65 (2021-04-14—2026-09-01)


In [2]:
# ---------- ЧАСТИНА 1: аудит негативної затримки (kev_date_added < published) ----------
has_both = master["published"].notna() & master["kev_date_added"].notna()
neg_lag = master[has_both & (master["kev_date_added"] < master["published"])].copy()
neg_lag["lag_days"] = (neg_lag["kev_date_added"] - neg_lag["published"]).dt.days

print(f"CVE з обома датами: {has_both.sum():,}")
print(f"З них негативна затримка (kev_date_added < published): {len(neg_lag):,} "
      f"({len(neg_lag) / has_both.sum():.1%})")

print("\nРозподіл kev_date_added серед аномальних записів (перевірка гіпотези про запуск каталогу):")
print(neg_lag["kev_date_added"].dt.to_period("M").value_counts().sort_index().head(15))

print("\nЧастка аномальних записів, доданих САМЕ в місяць запуску каталогу (2021-11) або раніше:")
launch_month_share = (neg_lag["kev_date_added"] <= pd.Timestamp("2021-11-30")).mean()
print(f"  {launch_month_share:.1%}")

print("\nРозподіл величини |lag_days| (наскільки KEV-дата випереджає published):")
print(neg_lag["lag_days"].abs().describe())

print("\nРозподіл vuln_status серед аномальних (якщо колонка присутня):")
if "vuln_status" in neg_lag.columns:
    print(neg_lag["vuln_status"].value_counts())
else:
    print("  (колонки vuln_status немає в master)")

print("\n5 прикладів з найбільшою за модулем аномалією:")
cols = [c for c in ["cve_id", "published", "kev_date_added", "lag_days"] if c in neg_lag.columns]
display(neg_lag.reindex(neg_lag["lag_days"].abs().sort_values(ascending=False).index)[cols].head(5))

CVE з обома датами: 1,716
З них негативна затримка (kev_date_added < published): 223 (13.0%)

Розподіл kev_date_added серед аномальних записів (перевірка гіпотези про запуск каталогу):
kev_date_added
2021-11    2
2021-12    5
2022-01    1
2022-02    3
2022-03    3
2022-04    4
2022-05    2
2022-06    1
2022-07    1
2022-08    5
2022-09    7
2022-10    4
2022-11    4
2022-12    4
2023-01    1
Freq: M, Name: count, dtype: int64

Частка аномальних записів, доданих САМЕ в місяць запуску каталогу (2021-11) або раніше:
  0.9%

Розподіл величини |lag_days| (наскільки KEV-дата випереджає published):
count    223.000000
mean       9.147982
std       35.855573
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max      291.000000
Name: lag_days, dtype: float64

Розподіл vuln_status серед аномальних (якщо колонка присутня):
vuln_status
Analyzed    223
Name: count, dtype: int64

5 прикладів з найбільшою за модулем аномалією:


,cve_id,published,kev_date_added,lag_days
202783,CVE-2022-26486,2022-12-22 20:15:22.797,2022-03-07,-291
202782,CVE-2022-26485,2022-12-22 20:15:22.563,2022-03-07,-291
209103,CVE-2019-8720,2023-03-06 23:15:10.287,2022-05-23,-288
191612,CVE-2022-1096,2022-07-23 00:15:08.333,2022-03-28,-118
191797,CVE-2022-1364,2022-07-26 22:15:09.147,2022-04-15,-103


In [3]:
# ---------- ЧАСТИНА 1 (продовження): категоризація для подальшого аналізу (без видалення) ----------
master["temporal_consistency"] = "consistent"
master.loc[has_both & (master["kev_date_added"] < master["published"]), "temporal_consistency"] = "anomalous"
print(master["temporal_consistency"].value_counts())
print("\nℹ Ці 279 записів НЕ видаляються, а зберігаються з поміткою 'anomalous' у фінальному")
print("  датасеті (для прозорості в методології). Для Step 4A.2 нижче точка відліку t визначається")
print("  як перший ДОСТУПНИЙ знімок EPSS на/після published — якщо kev_date_added < published,")
print("  ця CVE вже буде 'already_kev' на момент t за побудовою (у неї немає шансу стати")
print("  'майбутньою' подією), і піде в категорію already_kev, а не поламає розрахунок.")

temporal_consistency
consistent    395516
anomalous        223
Name: count, dtype: int64

ℹ Ці 279 записів НЕ видаляються, а зберігаються з поміткою 'anomalous' у фінальному
  датасеті (для прозорості в методології). Для Step 4A.2 нижче точка відліку t визначається
  як перший ДОСТУПНИЙ знімок EPSS на/після published — якщо kev_date_added < published,
  ця CVE вже буде 'already_kev' на момент t за побудовою (у неї немає шансу стати
  'майбутньою' подією), і піде в категорію already_kev, а не поламає розрахунок.


In [4]:
# ---------- ЧАСТИНА 2: temporal CVE forecast dataset — ОДНА CVE = ОДИН РЯДОК ----------
# Точка відліку t для кожної CVE: ПЕРШИЙ знімок EPSS на дату >= published. Це природний
# момент, коли CVE вперше стає "видимою" для оцінки ризику. Одна CVE -> один рядок:
# усуває саму можливість витоку через появу тієї самої CVE в сусідніх знімках/спліт-межах.
HORIZONS_MAIN = [180, 365]   # 180д — primary, 365д — sensitivity (за результатом аудиту)

snap_arr = np.array(snap_dates)

def first_snapshot_on_or_after(d):
    if pd.isna(d):
        return pd.NaT
    idx = np.searchsorted(snap_arr, d, side="left")
    return snap_arr[idx] if idx < len(snap_arr) else pd.NaT

# ---- КРИТИЧНЕ ВИПРАВЛЕННЯ: покриття EPSS ЗРОСТАЄ з часом (65-81 тис. CVE у 2021 -> 380+ тис.
# у 2026), незалежно від дати публікації CVE. "Перший знімок НА/ПІСЛЯ published" НЕ гарантує,
# що САМЕ ЦЯ CVE вже оцінювалась EPSS у той момент — вона могла потрапити в покриття значно
# пізніше. Тому t_ref має враховувати ФАКТИЧНУ появу CVE в EPSS, а не лише календар.
# Будуємо один раз глобальний індекс "перша дата, коли ця CVE з'явилась у ЯКОМУСЬ знімку EPSS".
import gzip, re as _re
print("Будуємо індекс першої появи кожної CVE в EPSS (один прохід по всіх знімках)...")
first_epss_appearance = {}
for t, p in zip(snap_dates, snaps):
    raw = pd.read_csv(p, comment="#", usecols=[0], names=None, header=0)
    raw.columns = [str(c).strip().lower() for c in raw.columns]
    cve_col = "cve" if "cve" in raw.columns else raw.columns[0]
    for cve_id in raw[cve_col]:
        if cve_id not in first_epss_appearance:   # snap_dates за зростанням -> перша зустріч і є найранішою
            first_epss_appearance[cve_id] = t
print(f"Індекс побудовано: {len(first_epss_appearance):,} унікальних CVE коли-небудь мали EPSS-оцінку")

master["first_epss_appearance"] = master["cve_id"].map(first_epss_appearance)
n_never_scored = master["first_epss_appearance"].isna().sum()
print(f"CVE, що НІКОЛИ не з'являлись у жодному знімку EPSS: {n_never_scored:,} "
      f"({n_never_scored / len(master):.1%}) — виключаються (немає EPSS(t) в принципі)")

calendar_t = master["published"].map(first_snapshot_on_or_after)
master["t_ref"] = pd.concat([calendar_t, master["first_epss_appearance"]], axis=1).max(axis=1, skipna=False)
n_no_ref = master["t_ref"].isna().sum()
print(f"\nCVE без доступного t_ref (немає ні календарного знімка, ні появи в EPSS): {n_no_ref:,} "
      f"— виключаються з датасету")

ds = master[master["t_ref"].notna()].copy()
ds["epss_lag_days"] = (ds["t_ref"] - ds["published"]).dt.days

print("\nРозподіл epss_lag_days (наскільки t_ref пізніший за published) — НЕ фільтруємо за цим,")
print("лише перевіряємо, наскільки 't' реально близький до моменту розкриття вразливості:")
print(ds["epss_lag_days"].describe())
bins = [-1, 0, 7, 30, 90, 365, 1e9]
labels = ["=0д", "1-7д", "8-30д", "31-90д", "91-365д", ">365д"]
print("\nРозподіл по бакетах:")
print(pd.cut(ds["epss_lag_days"], bins=bins, labels=labels).value_counts().sort_index())
print("\n(великий lag означає, що CVE опублікована МІЖ доступними щомісячними знімками EPSS,")
print(" або опублікована ЗАДОВГО до старту EPSS-історії для pre-EPSS CVE)")

# ---- КРИТИЧНЕ РОЗРІЗНЕННЯ: для CVE до старту EPSS (2021-04-14) epss_lag_days відображає
# НЕ "затримку оцінки після розкриття", а "затримку ретроспективної доступності" —
# зовсім інший смисл, і його треба явно позначити окремим прапорцем, а не лишати
# прихованим усередині одного суцільного розподілу.
FIRST_EPSS_EVER = snap_dates[0]
ds["post_epss_published"] = ds["published"] >= FIRST_EPSS_EVER
print(f"\nCVE, опубліковані ПІСЛЯ старту EPSS ({FIRST_EPSS_EVER.date()}): "
      f"{int(ds['post_epss_published'].sum()):,} ({ds['post_epss_published'].mean():.1%})")
print(f"CVE, опубліковані ДО старту EPSS (retrospective availability lag, не delay-до-оцінки): "
      f"{int((~ds['post_epss_published']).sum()):,} ({(~ds['post_epss_published']).mean():.1%})")

print("\nepss_lag_days ОКРЕМО для post-EPSS CVE (тут це справді 'затримка до першого знімка'):")
print(ds.loc[ds["post_epss_published"], "epss_lag_days"].describe())
print("\nepss_lag_days ОКРЕМО для pre-EPSS CVE (тут це 'ретроспективна доступність', НЕ затримка оцінки):")
print(ds.loc[~ds["post_epss_published"], "epss_lag_days"].describe())

# ТОЧНЕ формулювання для методології (замінює раніше надто оптимістичне "lag має бути малим,
# ~0-30 днів"): переважна більшість post-EPSS CVE справді мають малий lag, але помітна
# частка (див. Діагностику B нижче) — ні, і це системно пов'язано з неповними метаданими.
print("\nℹ Для методології: 'For post-EPSS CVEs, the first available EPSS snapshot is typically")
print("  close to the publication date (median ~16 days; IQR 8-23 days), but a small subset")
print("  (~1.16%) exhibits substantially longer availability lags, primarily among records")
print("  with incomplete vulnerability metadata (see Diagnostic B).'")

# already_kev НА МОМЕНТ t_ref — виключаємо з кандидатів (немає шансу стати "майбутньою" подією)
already_kev_at_t = ds["kev_date_added"].notna() & (ds["kev_date_added"] <= ds["t_ref"])
n_already_post = int((already_kev_at_t & ds["post_epss_published"]).sum())
n_already_pre = int((already_kev_at_t & ~ds["post_epss_published"]).sum())
print(f"Із {len(ds):,} CVE з t_ref: вже в KEV на момент t_ref — {int(already_kev_at_t.sum()):,} (виключено), "
      f"з них post-EPSS: {n_already_post:,}, pre-EPSS: {n_already_pre:,}")
n_post_before = int(ds["post_epss_published"].sum())
ds = ds[~already_kev_at_t].copy()
n_post_after = int(ds["post_epss_published"].sum())
print(f"Залишилось кандидатів (одна CVE = один рядок): {len(ds):,}")
print(f"\nℹ АРИФМЕТИЧНА ЗВІРКА post-EPSS підмножини (для прозорості, щоб уникнути враження")
print(f"  'нової бази без пояснення' у подальших підрахунках):")
print(f"  post-EPSS до фільтра already_kev:  {n_post_before:,}")
print(f"  − виключено (already_kev, post-EPSS): {n_already_post:,}")
print(f"  = post-EPSS після фільтра:          {n_post_after:,}")
assert n_post_before - n_already_post == n_post_after, "арифметика не сходиться — перевірити вручну"
print("  ✅ арифметика сходиться")

for H in HORIZONS_MAIN:
    window_end = ds["t_ref"] + pd.Timedelta(days=H)
    kev = ds["kev_date_added"]
    is_pos = kev.notna() & (kev > ds["t_ref"]) & (kev <= window_end)
    is_known_neg = kev.notna() & (kev > window_end)
    is_nan = kev.isna()
    observed = window_end <= TODAY
    is_neg = is_known_neg | (is_nan & observed)
    is_cens = is_nan & ~observed
    ds[f"future_kev_{H}d"] = np.select([is_pos, is_neg, is_cens], [1, 0, np.nan], default=np.nan)
    ds[f"censored_{H}d"] = is_cens
    # категоріальний статус поруч із числовим таргетом — усуває плутанину, коли в одному
    # parquet лежать результати для кількох горизонтів одночасно (already_kev тут не
    # з'явиться: такі рядки вже виключені з ds на попередньому кроці, до цього циклу)
    ds[f"future_kev_status_{H}d"] = np.select([is_pos, is_neg, is_cens],
                                               ["future_kev", "no_future_kev", "right_censored"],
                                               default="?")
    n_obs = int((~is_cens).sum())
    n_pos = int(is_pos.sum())
    print(f"\nH={H}д: {n_obs:,} спостережуваних ({int(is_cens.sum()):,} цензуровано), "
          f"{n_pos:,} позитивних ({n_pos / n_obs:.4%} серед спостережуваних)")

Будуємо індекс першої появи кожної CVE в EPSS (один прохід по всіх знімках)...
Індекс побудовано: 379,591 унікальних CVE коли-небудь мали EPSS-оцінку
CVE, що НІКОЛИ не з'являлись у жодному знімку EPSS: 16,148 (4.1%) — виключаються (немає EPSS(t) в принципі)

CVE без доступного t_ref (немає ні календарного знімка, ні появи в EPSS): 16,211 — виключаються з датасету

Розподіл epss_lag_days (наскільки t_ref пізніший за published) — НЕ фільтруємо за цим,
лише перевіряємо, наскільки 't' реально близький до моменту розкриття вразливості:
count    379528.000000
mean       1152.199258
std        1929.790013
min           0.000000
25%          14.000000
50%          26.000000
75%        1432.000000
max       12203.000000
Name: epss_lag_days, dtype: float64

Розподіл по бакетах:
epss_lag_days
=0д          6188
1-7д        44989
8-30д      161420
31-90д       7899
91-365д     14141
>365д      144891
Name: count, dtype: int64

(великий lag означає, що CVE опублікована МІЖ доступними щомісячними зні

In [5]:
# ---------- ЧАСТИНА 2 (продовження): темпоральний спліт за t_ref, з урахуванням право-цензурування ----------
# Межі спліту — за датою t_ref (не випадково по рядках). Останній придатний t_ref для тесту
# кожного горизонту рахується ПРОГРАМНО (t_ref + H <= TODAY), а не вгадується вручну.
TRAIN_END = pd.Timestamp("2024-07-01")
VAL_END   = pd.Timestamp("2025-01-01")

def split_name(t_ref, H):
    last_usable_test = TODAY - pd.Timedelta(days=H)
    if t_ref > last_usable_test:
        return "excluded_censored"   # тестове вікно для цього t_ref ще не повністю спостережуване
    if t_ref < TRAIN_END:
        return "train"
    if t_ref < VAL_END:
        return "val"
    return "test"

for H in HORIZONS_MAIN:
    ds[f"split_{H}d"] = ds["t_ref"].map(lambda t: split_name(t, H))
    print(f"\nH={H}д — розподіл по спліту:")
    print(ds[f"split_{H}d"].value_counts())
    print("Позитивів по спліту:")
    valid = ds[ds[f"split_{H}d"] != "excluded_censored"]
    print(valid.groupby(f"split_{H}d")[f"future_kev_{H}d"].agg(["count", "sum", "mean"]))


H=180д — розподіл по спліту:
split_180d
train                249728
test                  65163
excluded_censored     48960
val                   15192
Name: count, dtype: int64
Позитивів по спліту:
             count    sum      mean
split_180d                         
test         65163   90.0  0.001381
train       249728  366.0  0.001466
val          15192   27.0  0.001777

H=365д — розподіл по спліту:
split_365d
train                249728
excluded_censored     74998
test                  39125
val                   15192
Name: count, dtype: int64
Позитивів по спліту:
             count    sum      mean
split_365d                         
test         39125   68.0  0.001738
train       249728  773.0  0.003095
val          15192   33.0  0.002172


In [6]:
# ---------- EPSS(t_ref) JOIN: реальне значення на момент t_ref кожної CVE, з метаданими ----------
# t_ref КОЖНОЇ CVE — це вже сам знімок EPSS (за побудовою: перший знімок на/після published),
# тому epss_t для будь-якого рядка — це справжнє, реально існуюче значення НА ЦЮ дату, а не
# "ретроспективна підстановка" в сенсі Step 3 (там ми брали найранiший знімок як stand-in для
# ЗОВСІМ ІНШОЇ дати мережевого потоку). Тут t_ref і дата знімка — те саме число.
epss_by_snapshot = {}   # snapshot_date -> {cve_id: (epss, model_version)}

def load_epss_snapshot(t):
    if t in epss_by_snapshot:
        return epss_by_snapshot[t]
    idx = snap_dates.index(t)
    p = snaps[idx]
    raw = pd.read_csv(p, comment="#")
    raw.columns = [str(c).strip().lower() for c in raw.columns]
    cve_col = "cve" if "cve" in raw.columns else raw.columns[0]
    epss_col = "epss" if "epss" in raw.columns else next((c for c in raw.columns if "epss" in c), raw.columns[1])
    mapping = dict(zip(raw[cve_col], raw[epss_col]))
    ver_match = None
    with gzip.open(p, "rt", errors="replace") as f:
        import re as _re
        ver_match = _re.search(r"model_version:(v[\d.]+)", f.readline())
    epss_by_snapshot[t] = (mapping, ver_match.group(1) if ver_match else "v1?")
    return epss_by_snapshot[t]

import gzip
epss_t_vals, epss_model_vals = [], []
for t, cve_id in zip(ds["t_ref"], ds["cve_id"]):
    mapping, ver = load_epss_snapshot(t)
    epss_t_vals.append(mapping.get(cve_id))
    epss_model_vals.append(ver)

ds["epss_t"] = epss_t_vals
ds["epss_snapshot_date"] = ds["t_ref"]   # синонім для ясності в майбутньому парquet — t_ref і є датою знімка
ds["epss_model_version"] = epss_model_vals

n_missing_epss = ds["epss_t"].isna().sum()
print(f"CVE без epss_t після join: {n_missing_epss:,} з {len(ds):,} ({n_missing_epss / len(ds):.2%})")
print("Причина (перевірено Діагностикою A): CVE з'явилась в EPSS ОДИН раз, а на власному t_ref")
print("(наступний знімок) вже ВІДСУТНЯ — тобто випала зі скорингу між сусідніми знімками")
print("(ймовірно, відкликана/змінена), а НЕ розбіжність формату cve_id.")

# ---- ОСТАТОЧНЕ ВИКЛЮЧЕННЯ: без валідного EPSS(t_ref) запис непридатний для задачі
# "яким був EPSS на момент t_ref" — не підставляємо значення з іншого знімка (це змінило б
# темпоральну семантику), а прямо виключаємо і документуємо частку.
# Зберігаємо самі відкинуті рядки ОКРЕМО (missing_epss_records) — вони знадобляться
# Діагностиці A нижче, яка інакше не знайшла б їх у вже відфільтрованому ds.
n_before_epss_filter = len(ds)
missing_epss_records = ds[ds["epss_t"].isna()].copy()
ds = ds[ds["epss_t"].notna()].copy()
print(f"\nВиключено {n_missing_epss:,} записів ({n_missing_epss / n_before_epss_filter:.3%}) без валідного "
      f"EPSS(t_ref) — задокументовано як unmatched observations, не виправлено підстановкою.")
print(f"Фінальний розмір датасету: {len(ds):,} рядків.")

CVE без epss_t після join: 7 з 379,043 (0.00%)
Причина (перевірено Діагностикою A): CVE з'явилась в EPSS ОДИН раз, а на власному t_ref
(наступний знімок) вже ВІДСУТНЯ — тобто випала зі скорингу між сусідніми знімками
(ймовірно, відкликана/змінена), а НЕ розбіжність формату cve_id.

Виключено 7 записів (0.002%) без валідного EPSS(t_ref) — задокументовано як unmatched observations, не виправлено підстановкою.
Фінальний розмір датасету: 379,036 рядків.


In [7]:
# ---------- ФІНАЛЬНИЙ АУДИТ-ЧЕКЛІСТ перед "заморожуванням" Step 4A ----------
print("=" * 70)
print("1. Точні межі t_ref по спліту (для таблиці в статті)")
print("=" * 70)
for H in HORIZONS_MAIN:
    valid = ds[ds[f"split_{H}d"] != "excluded_censored"]
    print(f"\nH={H}д:")
    tbl = valid.groupby(f"split_{H}d").agg(
        t_ref_min=("t_ref", "min"), t_ref_max=("t_ref", "max"),
        n=("cve_id", "count"), positives=(f"future_kev_{H}d", "sum"))
    display(tbl)

print("\n" + "=" * 70)
print("2. Дублікати cve_id (має бути 0 — одна CVE = один рядок)")
print("=" * 70)
n_dup = ds["cve_id"].duplicated().sum()
print(f"Дублікатів cve_id: {n_dup}", "✅" if n_dup == 0 else "❌ ПРОБЛЕМА")

print("\n" + "=" * 70)
print("3. Чи є CVE, присутня в різних спліт-сегментах одночасно (для будь-якого H)")
print("=" * 70)
for H in HORIZONS_MAIN:
    # тривіально неможливо за конструкцією (один рядок на CVE, один split на рядок),
    # але перевіряємо explicit, а не покладаємось на 'це не може статися за дизайном'
    n_multi = ds.groupby("cve_id")[f"split_{H}d"].nunique()
    bad = (n_multi > 1).sum()
    print(f"H={H}д: CVE у кількох спліт-сегментах одночасно: {bad}", "✅" if bad == 0 else "❌ ПРОБЛЕМА")

print("\n" + "=" * 70)
print("4. Ручна перевірка на кількох прикладах (2 позитивних, 2 негативних, 2 цензурованих)")
print("=" * 70)
H_check = 180
for status, n in [("future_kev", 2), ("no_future_kev", 2), ("right_censored", 2)]:
    sample = ds[ds[f"future_kev_status_{H_check}d"] == status].head(n)
    print(f"\n--- {status} (H={H_check}d) ---")
    display(sample[["cve_id", "published", "t_ref", "kev_date_added", f"future_kev_{H_check}d"]])

print("\n" + "=" * 70)
print("5. Частка anomalous (temporal_consistency) по спліту — чи не сконцентровані в одному")
print("=" * 70)
for H in HORIZONS_MAIN:
    valid = ds[ds[f"split_{H}d"] != "excluded_censored"]
    print(f"\nH={H}д:")
    print(valid.groupby(f"split_{H}d")["temporal_consistency"].value_counts(normalize=True))

1. Точні межі t_ref по спліту (для таблиці в статті)

H=180д:


,t_ref_min,t_ref_max,n,positives
split_180d,,,,
test,2025-01-01,2026-03-01,65159,90.0
train,2021-04-14,2024-06-01,249725,366.0
val,2024-07-01,2024-11-01,15192,27.0



H=365д:


,t_ref_min,t_ref_max,n,positives
split_365d,,,,
test,2025-01-01,2025-09-01,39125,68.0
train,2021-04-14,2024-06-01,249725,773.0
val,2024-07-01,2024-11-01,15192,33.0



2. Дублікати cve_id (має бути 0 — одна CVE = один рядок)
Дублікатів cve_id: 0 ✅

3. Чи є CVE, присутня в різних спліт-сегментах одночасно (для будь-якого H)
H=180д: CVE у кількох спліт-сегментах одночасно: 0 ✅
H=365д: CVE у кількох спліт-сегментах одночасно: 0 ✅

4. Ручна перевірка на кількох прикладах (2 позитивних, 2 негативних, 2 цензурованих)

--- future_kev (H=180d) ---


...


--- no_future_kev (H=180d) ---


,cve_id,published,t_ref,kev_date_added,future_kev_180d
0,CVE-1999-0095,1988-10-01 04:00:00,2022-03-01,NaT,0.0
1,CVE-1999-0082,1988-11-11 05:00:00,2022-03-01,NaT,0.0



--- right_censored (H=180d) ---


,cve_id,published,t_ref,kev_date_added,future_kev_180d
289804,CVE-2025-1782,2025-04-14 19:15:36.277,2026-06-01,NaT,NaN
308535,CVE-2025-58375,2025-09-06 00:15:35.047,2026-09-01,NaT,NaN



5. Частка anomalous (temporal_consistency) по спліту — чи не сконцентровані в одному

H=180д:
split_180d  temporal_consistency
test        consistent              1.0
train       consistent              1.0
val         consistent              1.0
Name: proportion, dtype: float64

H=365д:
split_365d  temporal_consistency
test        consistent              1.0
train       consistent              1.0
val         consistent              1.0
Name: proportion, dtype: float64


In [8]:
# ---------- ДІАГНОСТИКА A: розбір 7 CVE без epss_t після join ----------
missing = missing_epss_records   # збережено в комірці EPSS-join, ДО остаточного виключення з ds
print(f"CVE без epss_t: {len(missing)}")
display(missing[["cve_id", "published", "t_ref", "first_epss_appearance"]])

print("\nПеревірка кожної: чи є цей cve_id у СИРОМУ файлі знімка, що відповідає її власному t_ref")
print("(розрізняє: 'зникла з пізнішого знімка після появи в ранішому' від 'справжня помилка формату')")
for _, row in missing.iterrows():
    t = row["t_ref"]
    idx = snap_dates.index(t)
    p = snaps[idx]
    raw = pd.read_csv(p, comment="#", usecols=[0], header=0)
    raw.columns = [str(c).strip().lower() for c in raw.columns]
    cve_col = "cve" if "cve" in raw.columns else raw.columns[0]
    present_in_own_snapshot = row["cve_id"] in set(raw[cve_col])
    # чи були дублікати цього cve_id у файлі знімка t_ref (могло б зламати dict-мапінг)
    n_occurrences = int((raw[cve_col] == row["cve_id"]).sum())
    print(f"  {row['cve_id']}: t_ref={t.date()}  присутня_у_власному_знімку={present_in_own_snapshot}  "
          f"кількість_входжень_у_файлі={n_occurrences}")

CVE без epss_t: 7


,cve_id,published,t_ref,first_epss_appearance
177223,CVE-2021-45945,2022-01-01 00:15:08.230,2022-02-01,2022-01-01
177237,CVE-2021-45959,2022-01-01 00:15:08.860,2022-02-01,2022-01-01
181457,CVE-2021-35036,2022-03-01 07:15:06.917,2022-04-01,2022-03-01
312152,CVE-2025-39898,2025-10-01 08:15:32.483,2025-11-01,2025-10-01
312216,CVE-2022-50450,2025-10-01 12:15:37.680,2025-11-01,2025-10-01
312221,CVE-2022-50455,2025-10-01 12:15:38.657,2025-11-01,2025-10-01
312290,CVE-2023-53502,2025-10-01 12:15:53.770,2025-11-01,2025-10-01



Перевірка кожної: чи є цей cve_id у СИРОМУ файлі знімка, що відповідає її власному t_ref
(розрізняє: 'зникла з пізнішого знімка після появи в ранішому' від 'справжня помилка формату')
  CVE-2021-45945: t_ref=2022-02-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2021-45959: t_ref=2022-02-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2021-35036: t_ref=2022-04-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2025-39898: t_ref=2025-11-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2022-50450: t_ref=2025-11-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2022-50455: t_ref=2025-11-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0
  CVE-2023-53502: t_ref=2025-11-01  присутня_у_власному_знімку=False  кількість_входжень_у_файлі=0


In [9]:
# ---------- ДІАГНОСТИКА B: топ post-EPSS CVE з найбільшим epss_lag_days (>60д — нетипово) ----------
post_epss_ds = ds[ds["post_epss_published"]].copy()
suspicious = post_epss_ds[post_epss_ds["epss_lag_days"] > 60].sort_values("epss_lag_days", ascending=False)
print(f"post-EPSS CVE з epss_lag_days > 60 днів: {len(suspicious):,} з {len(post_epss_ds):,} "
      f"({len(suspicious) / len(post_epss_ds):.2%})")

cols = [c for c in ["cve_id", "published", "t_ref", "epss_lag_days", "cvss_score", "cwe_ids", "vuln_status"]
        if c in suspicious.columns]
print("\nТоп-20 за величиною лагу:")
display(suspicious[cols].head(20))

print("\nРозподіл epss_lag_days серед 'підозрілих' (>60д) — чи це довгий хвіст, чи концентрація:")
print(suspicious["epss_lag_days"].describe())

print("\nЧи мають ці CVE CWE/CVSS (перевірка гіпотези 'бракувало метаданих для скорингу')?")
if "cwe_ids" in suspicious.columns:
    has_cwe = suspicious["cwe_ids"].map(lambda x: x is not None and len(x) > 0)
    print(f"  З CWE: {has_cwe.mean():.1%}")
if "cvss_score" in suspicious.columns:
    print(f"  З CVSS: {suspicious['cvss_score'].notna().mean():.1%}")

print("\nДля порівняння — те саме серед НЕпідозрілих post-EPSS CVE (lag<=60д):")
normal = post_epss_ds[post_epss_ds["epss_lag_days"] <= 60]
if "cwe_ids" in normal.columns:
    print(f"  З CWE: {normal['cwe_ids'].map(lambda x: x is not None and len(x) > 0).mean():.1%}")
if "cvss_score" in normal.columns:
    print(f"  З CVSS: {normal['cvss_score'].notna().mean():.1%}")

post-EPSS CVE з epss_lag_days > 60 днів: 2,533 з 217,831 (1.16%)

Топ-20 за величиною лагу:


,cve_id,published,t_ref,epss_lag_days,cvss_score,cwe_ids,vuln_status
161374,CVE-2021-30138,2021-04-15 16:15:14.110,2024-03-01,1050,NaN,[],Rejected
161508,CVE-2021-30031,2021-04-21 16:15:08.797,2024-03-01,1044,NaN,[],Rejected
161835,CVE-2017-20003,2021-04-23 16:15:07.880,2024-03-01,1042,NaN,[],Rejected
162109,CVE-2021-27802,2021-04-29 16:15:09.903,2024-03-01,1036,NaN,[],Rejected
162514,CVE-2021-32259,2021-05-07 15:15:07.547,2024-03-01,1028,NaN,[],Rejected
162591,CVE-2021-31877,2021-05-10 16:15:07.480,2024-03-01,1025,NaN,[],Rejected
163538,CVE-2008-5084,2021-05-26 22:15:07.607,2024-03-01,1009,NaN,[],Rejected
163540,CVE-2008-5509,2021-05-26 22:15:07.667,2024-03-01,1009,NaN,[],Rejected
163539,CVE-2008-5085,2021-05-26 22:15:07.637,2024-03-01,1009,NaN,[],Rejected
163537,CVE-2008-3523,2021-05-26 22:15:07.570,2024-03-01,1009,NaN,[],Rejected



Розподіл epss_lag_days серед 'підозрілих' (>60д) — чи це довгий хвіст, чи концентрація:
count    2533.000000
mean      623.198184
std       278.885897
min        61.000000
25%       290.000000
50%       750.000000
75%       801.000000
max      1050.000000
Name: epss_lag_days, dtype: float64

Чи мають ці CVE CWE/CVSS (перевірка гіпотези 'бракувало метаданих для скорингу')?
  З CWE: 26.6%
  З CVSS: 29.3%

Для порівняння — те саме серед НЕпідозрілих post-EPSS CVE (lag<=60д):
  З CWE: 91.8%
  З CVSS: 98.3%


In [10]:
# ---------- ЗБЕРЕЖЕННЯ (фінальна версія, з EPSS-join і status-полями) ----------
out_cols = ["cve_id", "published", "t_ref", "epss_lag_days", "post_epss_published",
            "temporal_consistency", "epss_t", "epss_snapshot_date", "epss_model_version",
            "cvss_score", "cwe_ids", "products", "description"] + \
           [f"future_kev_{H}d" for H in HORIZONS_MAIN] + \
           [f"future_kev_status_{H}d" for H in HORIZONS_MAIN] + \
           [f"censored_{H}d" for H in HORIZONS_MAIN] + \
           [f"split_{H}d" for H in HORIZONS_MAIN]
out_cols = [c for c in out_cols if c in ds.columns]
out_path = Path("/srv/ids_research/data/cti/cve_forecast_dataset.parquet")
ds[out_cols].to_parquet(out_path, index=False)
print(f"Збережено: {out_path} ({len(ds):,} рядків, {len(out_cols)} колонок)")
print("\nКожна CVE — рівно один рядок, з t_ref власним для кожної CVE.")
print("EPSS(t_ref) домержено з відповідного знімка (epss_t), разом із моделлю (epss_model_version)")
print("і прапорцем post_epss_published — чи CVE опублікована після старту EPSS-історії.")


Збережено: /srv/ids_research/data/cti/cve_forecast_dataset.parquet (379,036 рядків, 21 колонок)

Кожна CVE — рівно один рядок, з t_ref власним для кожної CVE.
EPSS(t_ref) домержено з відповідного знімка (epss_t), разом із моделлю (epss_model_version)
і прапорцем post_epss_published — чи CVE опублікована після старту EPSS-історії.
